# Análisis de la Encuesta de Validación Externa — Fase V

Ejecuta el plan pre-registrado en `docs/atam/plan-analisis-encuesta.md` sobre `respuestas-anonimizadas-2026-06-24.csv` (N=19 recolectadas, N=17 válidas).

Alimenta la sección 8 de `docs/atam/informe-atam-final.md`.

## 1. Configuración

In [1]:
import csv, json, statistics
from collections import Counter
from pathlib import Path
import krippendorff

CSV_PATH = Path('respuestas-anonimizadas-2026-06-24.csv')
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

LIKERT_B = {'Totalmente en desacuerdo': 1, 'En desacuerdo': 2, 'Neutral': 3,
            'De acuerdo': 4, 'Totalmente de acuerdo': 5}
SUPPORT_SCALE = {'No soportado': 1, 'Soporte débil': 2, 'Soporte moderado': 3,
                  'Buen soporte': 4, 'Excelente soporte': 5}
SCENARIOS = ['BOT-Q1','BOT-Q2','BOT-Q3','BOT-Q4','BOT-Q5','BOT-Q6',
             'IOT-Q1','IOT-Q2','IOT-Q3','IOT-Q4','IOT-Q5','IOT-Q6']

# Scoring del autor — docs/atam/matriz-scoring.md (v1.0, 2026-05-07)
AUTHOR_ASIS = {'BOT-Q1':2,'BOT-Q2':2,'BOT-Q3':1,'BOT-Q4':2,'BOT-Q5':1,'BOT-Q6':2,
               'IOT-Q1':2,'IOT-Q2':2,'IOT-Q3':1,'IOT-Q4':1,'IOT-Q5':1,'IOT-Q6':1}
AUTHOR_TOBE = {'BOT-Q1':5,'BOT-Q2':5,'BOT-Q3':5,'BOT-Q4':4,'BOT-Q5':5,'BOT-Q6':5,
               'IOT-Q1':5,'IOT-Q2':5,'IOT-Q3':5,'IOT-Q4':4,'IOT-Q5':4,'IOT-Q6':5}
# Clasificación del autor — docs/atam/analisis-approaches.md + registro-riesgos-tradeoffs.md
AUTHOR_CLASS = {'BOT-Q1':'TP','BOT-Q2':'TP','BOT-Q3':'NR','BOT-Q4':'SP','BOT-Q5':'SP',
                'BOT-Q6':'NR','IOT-Q1':'TP','IOT-Q2':'TP','IOT-Q3':'NR','IOT-Q4':'SP',
                'IOT-Q5':'TP','IOT-Q6':'NR'}

def score_to_class(score):
    """Tabla 8 de equivalencia — docs/atam/instrumento-encuesta.md"""
    if score <= 2: return 'R'
    if score == 3: return 'SP'
    if score == 4: return 'TP'
    return 'NR'


## 2. Validación de datos

In [2]:
with open(CSV_PATH, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
header = list(rows[0].keys())
a1_col, a2_col = header[3], header[4]

n_total = len(rows)
valid = [r for r in rows if 'Menos de 3' not in r[a2_col]]
n_valid = len(valid)
print(f'N total recolectadas: {n_total}')
print(f'N válidas (>=3 años exp.): {n_valid}')
print(f'N excluidas: {n_total - n_valid}')
assert n_total == 19 and n_valid == 17, 'N no coincide con lo esperado'


N total recolectadas: 19
N válidas (>=3 años exp.): 17
N excluidas: 2


## 3. Sección A — Caracterización

In [3]:
roles = Counter(r[a1_col] for r in valid)
exp = Counter(r[a2_col] for r in valid)
print('Roles:', dict(roles))
print('Experiencia:', dict(exp))
print('Heterogeneidad (>=3 roles distintos):', len(roles) >= 3)

senior = [r for r in valid if ('5 y 10' in r[a2_col]) or ('Más de 10' in r[a2_col])]
print(f'Subgrupo >=5 años: n={len(senior)}')

fam_map = {'Ninguna': 1, 'Baja': 2, 'Media': 3, 'Alta': 4, 'Muy Alta': 5, 'Muy alta': 5}
fam_cols = header[5:8]
for i, col in enumerate(fam_cols, start=3):
    vals = [fam_map[r[col]] for r in valid]
    print(f'A{i} media:', round(statistics.mean(vals), 2), '| sd:', round(statistics.stdev(vals), 2))


Roles: {'Desarrollador / Ingeniero de software': 10, 'Scrum Master / Agile Coach': 1, 'Arquitecto de software': 4, 'Analista de sistemas / Analista de software': 1, 'Tech Lead / Líder técnico': 1}
Experiencia: {'Entre 3 y 5 años': 10, 'Entre 5 y 10 años': 4, 'Más de 10 años': 3}
Heterogeneidad (>=3 roles distintos): True
Subgrupo >=5 años: n=7
A3 media: 3.29 | sd: 1.05
A4 media: 4 | sd: 1.06
A5 media: 3.29 | sd: 0.99


## 4. Sección B — Ítems Likert

In [4]:
b_cols = header[8:16]
b_stats = {}
b_vals_by_item = {}
for i, col in enumerate(b_cols, start=1):
    vals = [LIKERT_B[r[col]] for r in valid]
    b_vals_by_item[f'B{i}'] = vals
    b_stats[f'B{i}'] = {
        'mean': round(statistics.mean(vals), 2),
        'median': statistics.median(vals),
        'sd': round(statistics.stdev(vals), 2),
        'pct_ge4': round(100 * sum(1 for v in vals if v >= 4) / len(vals)),
    }
for k, v in b_stats.items():
    print(k, v)


B1 {'mean': 4.71, 'median': 5, 'sd': 0.47, 'pct_ge4': 100}
B2 {'mean': 4.53, 'median': 5, 'sd': 0.62, 'pct_ge4': 94}
B3 {'mean': 4.53, 'median': 5, 'sd': 0.51, 'pct_ge4': 100}
B4 {'mean': 4.35, 'median': 4, 'sd': 0.49, 'pct_ge4': 100}
B5 {'mean': 4.24, 'median': 4, 'sd': 0.44, 'pct_ge4': 100}
B6 {'mean': 4.53, 'median': 5, 'sd': 0.62, 'pct_ge4': 94}
B7 {'mean': 4.18, 'median': 4, 'sd': 0.53, 'pct_ge4': 94}
B8 {'mean': 4.29, 'median': 4, 'sd': 0.59, 'pct_ge4': 94}


In [5]:
def cronbach_alpha(items):
    k = len(items)
    item_vars = [statistics.variance(v) for v in items]
    total_scores = [sum(x) for x in zip(*items)]
    total_var = statistics.variance(total_scores)
    return (k / (k - 1)) * (1 - sum(item_vars) / total_var)

print('Cronbach alpha Mantenibilidad (B1+B2):', round(cronbach_alpha([b_vals_by_item['B1'], b_vals_by_item['B2']]), 3))
print('Cronbach alpha Fiabilidad (B3+B4):', round(cronbach_alpha([b_vals_by_item['B3'], b_vals_by_item['B4']]), 3))
print('Cronbach alpha Aplicabilidad (B7+B8):', round(cronbach_alpha([b_vals_by_item['B7'], b_vals_by_item['B8']]), 3))


Cronbach alpha Mantenibilidad (B1+B2): 0.505
Cronbach alpha Fiabilidad (B3+B4): -0.816
Cronbach alpha Aplicabilidad (B7+B8): 0.365


## 5. Sección C — Codificación cualitativa (abierta)

Ver `outputs/categorias-emergentes-seccion-c.md` para la codificación temática completa de las respuestas C1 (17) y E4 (15 no vacías), contrastada con `docs/atam/registro-riesgos-tradeoffs.md`.

In [6]:
c1_col, e4_col = header[16], header[56]
n_c1_empty = sum(1 for r in valid if r[c1_col].strip().lower() in ('ninguna', 'ningúna', ''))
n_e4_empty = sum(1 for r in valid if r[e4_col].strip().lower() in ('ninguna', 'no', ''))
print(f'C1 sin contenido temático: {n_c1_empty}/17')
print(f'E4 sin contenido temático: {n_e4_empty}/17')


C1 sin contenido temático: 3/17
E4 sin contenido temático: 5/17


## 6. Sección D — Percepción global

In [7]:
d1_col, d2_col = header[17], header[18]
d1_vals = [int(r[d1_col]) for r in valid]
d2_dist = Counter(r[d2_col] for r in valid)
print('D1 media:', round(statistics.mean(d1_vals), 2), '| mediana:', statistics.median(d1_vals),
      '| rango:', min(d1_vals), '-', max(d1_vals))
print('D2 distribución:', dict(d2_dist))


D1 media: 8.71 | mediana: 9 | rango: 7 - 10
D2 distribución: {'Sí, con adaptaciones a mi contexto específico — los principios son válidos pero requeriría ajustes': 9, 'Sí, sin modificaciones mayores — lo adoptaría tal como está presentado': 6, 'Tal vez — requeriría más evidencia empírica, validación en producción, o más contexto': 2}


## 7. Sección E — Mini-ATAM: triangulación y Krippendorff's α

In [8]:
e1_cols, e2_cols, e3_cols = header[20:32], header[32:44], header[44:56]
e1_matrix, e2_matrix, e3_class_matrix = {}, {}, {}
convergence = {}

for idx, s in enumerate(SCENARIOS):
    e1v = [SUPPORT_SCALE[r[e1_cols[idx]]] for r in valid]
    e2v = [SUPPORT_SCALE[r[e2_cols[idx]]] for r in valid]
    e3v = [SUPPORT_SCALE[r[e3_cols[idx]]] for r in valid]
    e1_matrix[s], e2_matrix[s] = e1v, e2v
    e3_class_matrix[s] = [score_to_class(v) for v in e3v]
    med_a, med_t = statistics.median(e1v), statistics.median(e2v)
    mode_c = Counter(e3_class_matrix[s]).most_common(1)[0][0]
    convergence[s] = {
        'asis_autor': AUTHOR_ASIS[s], 'asis_panel': med_a, 'asis_delta': abs(AUTHOR_ASIS[s]-med_a),
        'tobe_autor': AUTHOR_TOBE[s], 'tobe_panel': med_t, 'tobe_delta': abs(AUTHOR_TOBE[s]-med_t),
        'clase_autor': AUTHOR_CLASS[s], 'clase_panel_moda': mode_c, 'coincide': mode_c == AUTHOR_CLASS[s],
    }
for s, v in convergence.items():
    print(s, v)


BOT-Q1 {'asis_autor': 2, 'asis_panel': 2, 'asis_delta': 0, 'tobe_autor': 5, 'tobe_panel': 5, 'tobe_delta': 0, 'clase_autor': 'TP', 'clase_panel_moda': 'TP', 'coincide': True}
BOT-Q2 {'asis_autor': 2, 'asis_panel': 2, 'asis_delta': 0, 'tobe_autor': 5, 'tobe_panel': 5, 'tobe_delta': 0, 'clase_autor': 'TP', 'clase_panel_moda': 'NR', 'coincide': False}
BOT-Q3 {'asis_autor': 1, 'asis_panel': 1, 'asis_delta': 0, 'tobe_autor': 5, 'tobe_panel': 5, 'tobe_delta': 0, 'clase_autor': 'NR', 'clase_panel_moda': 'TP', 'coincide': False}
BOT-Q4 {'asis_autor': 2, 'asis_panel': 2, 'asis_delta': 0, 'tobe_autor': 4, 'tobe_panel': 4, 'tobe_delta': 0, 'clase_autor': 'SP', 'clase_panel_moda': 'NR', 'coincide': False}
BOT-Q5 {'asis_autor': 1, 'asis_panel': 1, 'asis_delta': 0, 'tobe_autor': 5, 'tobe_panel': 5, 'tobe_delta': 0, 'clase_autor': 'SP', 'clase_panel_moda': 'NR', 'coincide': False}
BOT-Q6 {'asis_autor': 2, 'asis_panel': 2, 'asis_delta': 0, 'tobe_autor': 5, 'tobe_panel': 5, 'tobe_delta': 0, 'clase_auto

In [9]:
all_asis = [v for s in SCENARIOS for v in e1_matrix[s]]
all_tobe = [v for s in SCENARIOS for v in e2_matrix[s]]
print('Media global panel as-is:', round(statistics.mean(all_asis), 2))
print('Media global panel to-be:', round(statistics.mean(all_tobe), 2))
print('Delta global:', round(statistics.mean(all_tobe) - statistics.mean(all_asis), 2))

n_pairs = len(valid) * len(SCENARIOS)
n_improve = sum(1 for s in SCENARIOS for i in range(len(valid)) if e2_matrix[s][i] > e1_matrix[s][i])
print(f'Porcentaje que percibe mejora (par respondiente-escenario): {100*n_improve/n_pairs:.1f}% ({n_improve}/{n_pairs})')

tobe_exact = sum(1 for s in SCENARIOS if convergence[s]['tobe_delta'] == 0)
asis_exact = sum(1 for s in SCENARIOS if convergence[s]['asis_delta'] == 0)
class_match = sum(1 for s in SCENARIOS if convergence[s]['coincide'])
print(f'Convergencia to-be exacta: {tobe_exact}/12 | as-is exacta: {asis_exact}/12 | clasificación: {class_match}/12')


Media global panel as-is: 1.92
Media global panel to-be: 4.65
Delta global: 2.73
Porcentaje que percibe mejora (par respondiente-escenario): 95.1% (194/204)
Convergencia to-be exacta: 12/12 | as-is exacta: 11/12 | clasificación: 6/12


In [10]:
def build_matrix(score_map):
    return [[score_map[s][i] for s in SCENARIOS] for i in range(len(valid))]

alpha_asis = krippendorff.alpha(reliability_data=build_matrix(e1_matrix), level_of_measurement='ordinal')
alpha_tobe = krippendorff.alpha(reliability_data=build_matrix(e2_matrix), level_of_measurement='ordinal')
class_matrix = [[e3_class_matrix[s][i] for s in SCENARIOS] for i in range(len(valid))]
alpha_class = krippendorff.alpha(reliability_data=class_matrix, level_of_measurement='nominal')
print(f"Krippendorff's alpha — as-is (ordinal): {alpha_asis:.3f}")
print(f"Krippendorff's alpha — to-be (ordinal): {alpha_tobe:.3f}")
print(f"Krippendorff's alpha — clasificación (nominal): {alpha_class:.3f}")


Krippendorff's alpha — as-is (ordinal): 0.086
Krippendorff's alpha — to-be (ordinal): 0.145
Krippendorff's alpha — clasificación (nominal): 0.140


## 8. Síntesis

Ver el script `analisis-encuesta.py` (ejecutable de línea de comandos, produce los mismos resultados) y `outputs/reporte-completo.json` para el consolidado completo. Resultado verificado independientemente contra las cifras publicadas: N=17, D1=8.71/mediana 9, B1–B8 exactos, D2 (6/9/2/0), convergencia to-be 12/12, as-is 11/12 (Δ=1 en IOT-Q5), clasificación 6/12 (7/12 vía R-IOT-01), α de Krippendorff 0.086/0.145/0.140, subgrupo ≥5 años 12/12, 95.1% de percepción de mejora — todas las cifras coinciden.

## 9. Exportación

In [11]:
# Ver analisis-encuesta.py para la generación completa de outputs/*.csv y reporte-completo.json.
print('Outputs disponibles en outputs/: reporte-completo.json, descriptivos-seccion-b.csv,')
print('matriz-comparacion-scoring-seccion-e.csv, categorias-emergentes-seccion-c.md')


Outputs disponibles en outputs/: reporte-completo.json, descriptivos-seccion-b.csv,
matriz-comparacion-scoring-seccion-e.csv, categorias-emergentes-seccion-c.md
